In [2]:
# ============================================================
# 1. IMPORT NECESSARY LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [3]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print("Dataset Shape:", X.shape)

print("\nFirst 5 Rows:")
print(X.head())

print("\nTarget Classes:")
print(data.target_names)

# 0 = malignant
# 1 = benign


Dataset Shape: (569, 30)

First 5 Rows:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst 

In [4]:
# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Set Shape:", X_train.shape)
print("Testing Set Shape :", X_test.shape)


Training Set Shape: (455, 30)
Testing Set Shape : (114, 30)


In [5]:
# ============================================================
# 4. STANDARDIZE FEATURES
# ============================================================

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("\nFeature scaling completed.")


Feature scaling completed.


In [6]:
# ============================================================
# 5. DEFINE SVM MODEL
# ============================================================

svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

In [7]:
# ============================================================
# 6. HYPERPARAMETER TUNING USING GRID SEARCH
# ============================================================

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.001, 0.0001]
}

grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train_sc, y_train)

print("\nBest Parameters from Grid Search:")
print(grid.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(f"{grid.best_score_:.4f}")



Best Parameters from Grid Search:
{'C': 10, 'gamma': 0.01}

Best Cross-Validation F1 Score:
0.9844


In [8]:
# ============================================================
# 7. TRAIN FINAL MODEL AND MAKE PREDICTIONS
# ============================================================

best_svm = grid.best_estimator_

best_svm.fit(X_train_sc, y_train)

y_pred = best_svm.predict(X_test_sc)

# Probability of class 1 (benign)
y_prob = best_svm.predict_proba(X_test_sc)[:, 1]

In [10]:
# ============================================================
# 8. EVALUATE THE MODEL
# ============================================================

acc = accuracy_score(y_test, y_pred)

prec = precision_score(
    y_test,
    y_pred,
    zero_division=0)

rec = recall_score(
    y_test,
    y_pred,
    zero_division=0)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0)

auc = roc_auc_score(
    y_test,
    y_prob)

cm = confusion_matrix(
    y_test,
    y_pred)
